![cassandra.png](https://cassandra.apache.org/assets/img/logo-white-r.png)

# TP01 — Modélisation Cassandra — **Solution**

Ce notebook reprend l'énoncé original et complète l'ensemble des questions
(modélisation, Docker, CQL). La partie CQL est exécutée depuis Python à
l'aide du driver officiel [`cassandra-driver`](https://docs.datastax.com/en/developer/python-driver/),
connecté au conteneur `cassandra` démarré par le `docker-compose.yml` fourni
à côté de ce notebook.

> 💡 Toutes les requêtes CQL sont volontairement exécutées une par une via une
> fonction utilitaire `run_cql(...)`, car le driver Cassandra n'autorise
> qu'une seule instruction CQL par appel `execute()` (contrairement à `cqlsh`
> qui peut enchaîner plusieurs instructions séparées par `;`).


# 1. Modélisation des Données

La modélisation des données est le processus d'identification des entités et de leurs relations. Dans les bases de données relationnelles, les données sont placées dans des bases normalisées tables avec des clés étrangères utilisées pour référencer des données connexes dans d'autres tables. Les requêtes que l'application va faire sont motivées par la structure de les tables et les données associées sont interrogées en tant que jointures de table.

Dans Cassandra, la modélisation des données est basée sur des requêtes. Les modèles d'accès aux données et les requêtes d'application déterminent la structure et l'organisation des données qui a ensuite utilisé pour concevoir les tables de base de données.

## Modélisation basée sur les requêtes
Contrairement à un modèle de base de données relationnelle dans lequel les requêtes utilisent table jointures pour obtenir des données de plusieurs tables, les jointures ne sont pas supportées Cassandra donc tous les champs obligatoires (colonnes) doivent être regroupés dans un une seule table. Comme chaque requête est soutenue par une table, les données sont dupliquées sur plusieurs tables dans un processus appelé dénormalisation. Données la duplication et un débit d'écriture élevé sont utilisés pour obtenir une lecture élevée performance
  

## Comparaison avec le Modèle de Données Relationnelles

Les bases de données relationnelles stockent des données dans des tables ayant des relations avec d'autres tables utilisant des clés étrangères.
Une base de données relationnelleApproche des données de l'annuaire la modélisation est centrée sur la table. Les requêtes doivent utiliser des jointures de table pour obtenir des données plusieurs tables qui ont une relation entre elles. Apache Cassandra le fait ne pas avoir le concept de clés étrangères ou d'intégrité relationnelle. Apache Le modèle de données de Cassandra’s est basé sur la conception de requêtes efficaces; requêtes qui ne’t impliquent plusieurs tables. Bases de données relationnelles normaliser les données pour éviter la duplication. Apache Cassandra en revanche dé-normalise les données en dupliquant les données dans plusieurs tables pour un modèle de données centré sur la requête. Si un modèle de données Cassandra ne peut pas être entièrement intégrer la complexité des relations entre les différentes entités pour une requête particulière, les jointures côté client dans le code d'application peuvent être utilisé.


# Modélisation relationelle vs colonnes

Lorsque vous envisagez de créer une nouvelle application pilotée par les données qui utilisera un base de données relationnelle, vous pouvez commencer par modéliser le domaine comme un ensemble de tables correctement normalisées et utiliser des clés étrangères pour référencer les tables liées données dans d'autres tableaux.

Cependant, quand on passe a une modélisation en colonnes il existe des differences :

* Pas de jointures
* Aucune intégrité référentielle
* Dénormalisation

La modélisation relationnelle, en termes simples, signifie que vous commencez par domaine conceptuel et ensuite représenter les noms dans le domaine dans les tables.

En revanche, dans Cassandra vous commencez avec le modèle de requête. Au lieu de modéliser les données d'abord et ensuite écrire des requêtes, avec Cassandra vous modélisez les requêtes et laissez les données être organisé autour d'eux. Pensez aux chemins de requête les plus courants l'application utilisera, puis créera les tables dont vous avez besoin soutenez-les.

## Exemple

### Realtionnelle

La figure ci-dessous montre comment vous pouvez représenter le stockage de données pour votre application utilisant un modèle de base de données relationnelle. Le modèle relationnel comprend un couple de tables join“” afin de réaliser les plusieurs-à-plusieurs les relations du modèle conceptuel des hôtels aux points d'intérêt chambres-à-aménités, chambres-à-disponibilité, et invités-à-chambres (via un réservation).

![r_design.png](https://cassandra.apache.org/doc/stable/cassandra/_images/data_modeling_hotel_relational.png)


### Conception orientée colonne

Essayons l'approche « query-first » pour commencer à concevoir le modèle de données d'une application hôtelière. La conception de l'interface utilisateur de l'application est souvent un excellent artefact à utiliser pour commencer à identifier les requêtes. Supposons que vous ayez discuté avec les parties prenantes du projet et que vos concepteurs UX aient produit des conceptions d'interface utilisateur ou des wireframes pour les principaux cas d'utilisation. Vous disposerez probablement d'une liste de requêtes d'achat comme la suivante :

- Q1. Trouver des hôtels à proximité d'un point d'intérêt donné.

- Q2. Trouver des informations sur un hôtel donné, telles que son nom et son emplacement.

- Q3. Trouver des points d'intérêt à proximité d'un hôtel donné.

- Q4. Trouver une chambre disponible dans une période donnée.

- Q5. Trouver le tarif et les équipements d'une chambre.


Maintenant que on a défini les requêtes, vous êtes prêt à commencer à concevoir Cassandra tables. Tout d'abord, créez un modèle logique contenant une table pour chaque requête, capture des entités et des relations à partir du conceptuel modèle.`

Pour nommer chaque table, vous identifierez le type d'entité primaire pour lequel vous effectuez une requête et l'utiliserez pour commencer le nom de l'entité. Si la requête porte sur les attributs d'autres entités apparentées, ajoutez-les au nom de la table, en les séparant par « by ». Par exemple, ``hotels_by_poi`` (POI pour *point of interst*).

Ensuite, vous identifiez la clé primaire de la table, en ajoutant la clé de partition colonnes basées sur les attributs de requête requis et colonnes de clustering afin de garantir l'unicité et de soutenir la commande de tri souhaitée.


### Résultat du modèle de Données Logiques Hôtel


La figure ci-dessous montre un modèle de données logique pour les requêtes impliquant des hôtels, des points d'intérêt, des chambres et des commodités. Une chose vous remarquerez immédiatement que le design de Cassandra ne comprend pas tables dédiées pour les chambres ou les équipements, comme vous l'aviez dans le relationnel conception. En effet, le flux de travail n'a identifié aucune requête nécessitant cet accès direct.

![result](https://cassandra.apache.org/doc/stable/cassandra/_images/data_modeling_hotel_logical.png)

Chaque tableau est affiché avec son titre et une liste de colonnes. Clé primaire les colonnes sont identifiées par des symboles tels que ``K`` pour les colonnes de clés de partition et ``C↑`` ou ``C↓`` pour représenter les colonnes de clustering. Lignes sont affichées entrer des tables ou entre les tables pour indiquer les requêtes que chacune la table est conçue pour supporter

# 2. A vous de jouer - Site de partage de vidéo  CassTube -

## 2.1. Modélisation

On souhaite modéliser la base de données pour le cas suivante :

Un site de partage de vidéo nommé CassTube, il a des utilisateurs possèdent un login unique, ainsi que leur nom et leur prenom. Les vidéos possèdent un id, un nom et une description. Un utilisateur possède plusieurs vidéos, et les vidéos possèdent des commentaires de plusieurs utilisateurs.

Nous souhaitons rechercher une liste des vidéos, ainsi que les vidéos par utilisateur. Enfin, nous chercherons aussi à récupérer les commentaires d'une vidéo ou les commentaires d'un utilisateurs

1. Proposer une modélisation normalisée (Par exemple avec UML ou E/A).
2. Maintenant, proposer une modélisation orienté colonne pour Cassandra.

## ✅ Réponse 2.1.1 — Modélisation normalisée (E/A)

**Entités et attributs**

| Entité  | Attributs | Clé |
|---|---|---|
| `User`    | login, firstname, lastname | PK : `login` |
| `Video`   | video_id, name, description, **owner_login** (FK → User) | PK : `video_id` |
| `Comment` | comment_id, **video_id** (FK → Video), **user_login** (FK → User), text, comment_date | PK : `comment_id` |

**Relations**

- `User (1,N) —— possède —— (1,1) Video` : un utilisateur possède plusieurs vidéos, une vidéo appartient à un seul utilisateur.
- `User (0,N) —— publie —— (1,1) Comment` : un utilisateur publie plusieurs commentaires.
- `Video (1,N) —— reçoit —— (1,1) Comment` : une vidéo reçoit plusieurs commentaires.

```mermaid
erDiagram
    USER ||--o{ VIDEO : possede
    USER ||--o{ COMMENT : publie
    VIDEO ||--o{ COMMENT : recoit

    USER {
        string login PK
        string firstname
        string lastname
    }
    VIDEO {
        int video_id PK
        string name
        string description
        string owner_login FK
    }
    COMMENT {
        int comment_id PK
        int video_id FK
        string user_login FK
        string text
        timestamp comment_date
    }
```

C'est un modèle **normalisé** classique : chaque information n'est stockée qu'à un seul endroit, et on reconstitue l'ensemble grâce à des jointures (`Video ⋈ User`, `Comment ⋈ Video ⋈ User`).

## ✅ Réponse 2.1.2 — Modélisation orientée colonne (Cassandra)

En Cassandra il n'y a pas de jointures : on part des **requêtes** exprimées dans l'énoncé et on crée **une table par requête**, en dénormalisant ce qu'il faut :

| # | Requête applicative | Table dédiée | Clé primaire |
|---|---|---|---|
| Q1 | Lister toutes les vidéos | `videos` | `PRIMARY KEY (video_id)` |
| Q2 | Vidéos d'un utilisateur donné | `user_videos_index` (alias `videos_by_user`) | `PRIMARY KEY (login, video_id)` |
| Q3 | Commentaires d'une vidéo donnée | `comments_by_video` | `PRIMARY KEY (video_id, user_login)` |
| Q4 | Commentaires d'un utilisateur donné | `comments_by_user` | `PRIMARY KEY (user_login, video_id)` |
| — | Infos de profil d'un utilisateur | `users` | `PRIMARY KEY (login)` |

On retrouve exactement les 5 tables utilisées plus bas dans le TP — c'est précisément ce schéma qui va être créé en CQL à la section 2.2.

⚠️ Remarque : c'est déjà de la **dénormalisation** — le login de l'auteur (`user_login`) est dupliqué dans `videos`, `user_videos_index`, `comments_by_video` et `comments_by_user`. C'est voulu et normal en Cassandra, mais cela impose une vigilance particulière côté application pour garder les données cohérentes (voir questions 4 et 5 plus loin).

## 2.2. Implémentation de la base de données sous Cassandra

### 🐳 Exécuter Cassandra avec Docker

Depuis la racine du dépôt, placez-vous dans `docker`. Démarrez le service et vérifiez son état :

:::bash
cd docker
docker compose up -d cassandra-tp
:::

Dans la suite du TP, vous allez pratiquer le CQL (Cassandra Query Language) en testant vos propres commandes.

### Interagir avec Cassandra en ligne de commande

```bash
docker exec -it cassandra-tp cqlsh
```

Les driver python pour cassandra ont été préalablement installés dans le notebook.


In [2]:
from cassandra.cluster import Cluster
import time


for attempt in range(10):
    try:
        cluster = Cluster(
            ['cassandra'],
            port=9042
        )
        session = cluster.connect()
        print("Connecté à Cassandra ✅")
        break
    except Exception as e:
        print(f"Cassandra pas encore prête ({e}), nouvelle tentative dans 5s...")
        time.sleep(5)

def run_cql(query, session=None, fetch=True):
    """Exécute UNE instruction CQL et affiche joliment le résultat / l'erreur."""
    s = session or globals()['session']
    print(f"CQL> {query.strip()}")
    try:
        result = s.execute(query)
        if fetch:
            rows = list(result)
            if rows:
                for row in rows:
                    print("  ", row)
            else:
                print("   (aucune ligne retournée / OK)")
        return result
    except Exception as e:
        print(f"   ⚠️  Erreur Cassandra : {e}")
        return None
    finally:
        print()


Cassandra pas encore prête (('Unable to connect to any servers', {'172.23.0.3:9042': ConnectionRefusedError(111, "Tried connecting to [('172.23.0.3', 9042)]. Last error: Connection refused")})), nouvelle tentative dans 5s...
Cassandra pas encore prête (('Unable to connect to any servers', {'172.23.0.3:9042': ConnectionRefusedError(111, "Tried connecting to [('172.23.0.3', 9042)]. Last error: Connection refused")})), nouvelle tentative dans 5s...


KeyboardInterrupt: 

### Executer les commandes CQL

vous pouvez consulter la documentation en ligne [Documentation officiel](https://cassandra.apache.org/doc/latest/cassandra/cql/index.html).  

1. Créer un nouveau keyspace nommé ``User_Video`` avec une option de replication ``NetworkTopologyStrategy``, et placez vous dans ce _keyspace_ :

**✅ Réponse Q1 (keyspace) + Q2 (tables)**

> ⚠️ Cassandra convertit automatiquement en minuscules les identifiants non
> quotés : le keyspace `User_Video` sera donc réellement stocké sous le nom
> `user_video` (vérifiable avec `DESCRIBE KEYSPACES;`). C'est ce nom en
> minuscules qui est utilisé partout ensuite.

```sql
CREATE KEYSPACE IF NOT EXISTS user_video
  WITH replication = {'class': 'NetworkTopologyStrategy', 'datacenter1': 1};

USE user_video;

CREATE TABLE videos (
    video_id        int,
    video_name      varchar,
    video_uploaded  timestamp,
    user_login      varchar,
    PRIMARY KEY (video_id)
);

CREATE TABLE users (
    login      varchar,
    firstname  varchar,
    lastname   varchar,
    PRIMARY KEY (login)
);

CREATE TABLE user_videos_index (
    login       varchar,
    video_id    int,
    video_name  varchar,
    PRIMARY KEY (login, video_id)
);

CREATE TABLE comments_by_video (
    video_id     int,
    user_login   varchar,
    comment_dat  timestamp,
    comment      varchar,
    PRIMARY KEY (video_id, user_login)
);

CREATE TABLE comments_by_user (
    user_login   varchar,
    video_id     int,
    comment_dat  timestamp,
    comment      varchar,
    PRIMARY KEY (user_login, video_id)
);
```

Le nommage suit la charte « `entité_by_critère` » présentée en partie 1 (ex. `hotels_by_poi`) : `comments_by_video` = commentaires **cherchés par** vidéo, `comments_by_user` = commentaires **cherchés par** utilisateur, `user_videos_index` = vidéos **cherchées par** utilisateur. Cela répond en même temps à la question 2 de l'énoncé (« créer les relations correspondantes ») : les 5 tables ci-dessus **sont** les tables demandées.

In [2]:
run_cql("CREATE KEYSPACE IF NOT EXISTS user_video "
        "WITH replication = {'class': 'NetworkTopologyStrategy', 'datacenter1': 1};")
session.set_keyspace('user_video')
print("Keyspace actif :", session.keyspace)


CQL> CREATE KEYSPACE IF NOT EXISTS user_video WITH replication = {'class': 'NetworkTopologyStrategy', 'datacenter1': 1};
   (aucune ligne retournée / OK)

Keyspace actif : user_video


2. Créer les relations correspondantes _video, user_videos_index ...__
(les types CQL sont disponibles à l’adresse https://cassandra.apache.org/doc/latest/cql/types.html)

**Hint :** Pour la charte de nommage regarder la question 3.


In [3]:
tables = [
    """CREATE TABLE IF NOT EXISTS videos (
        video_id        int,
        video_name      varchar,
        video_uploaded  timestamp,
        user_login      varchar,
        PRIMARY KEY (video_id)
    );""",
    """CREATE TABLE IF NOT EXISTS users (
        login      varchar,
        firstname  varchar,
        lastname   varchar,
        PRIMARY KEY (login)
    );""",
    """CREATE TABLE IF NOT EXISTS user_videos_index (
        login       varchar,
        video_id    int,
        video_name  varchar,
        PRIMARY KEY (login, video_id)
    );""",
    """CREATE TABLE IF NOT EXISTS comments_by_video (
        video_id     int,
        user_login   varchar,
        comment_dat  timestamp,
        comment      varchar,
        PRIMARY KEY (video_id, user_login)
    );""",
    """CREATE TABLE IF NOT EXISTS comments_by_user (
        user_login   varchar,
        video_id     int,
        comment_dat  timestamp,
        comment      varchar,
        PRIMARY KEY (user_login, video_id)
    );""",
]
for t in tables:
    run_cql(t)


CQL> CREATE TABLE IF NOT EXISTS videos (
        video_id        int,
        video_name      varchar,
        video_uploaded  timestamp,
        user_login      varchar,
        PRIMARY KEY (video_id)
    );
   (aucune ligne retournée / OK)

CQL> CREATE TABLE IF NOT EXISTS users (
        login      varchar,
        firstname  varchar,
        lastname   varchar,
        PRIMARY KEY (login)
    );
   (aucune ligne retournée / OK)

CQL> CREATE TABLE IF NOT EXISTS user_videos_index (
        login       varchar,
        video_id    int,
        video_name  varchar,
        PRIMARY KEY (login, video_id)
    );
   (aucune ligne retournée / OK)

CQL> CREATE TABLE IF NOT EXISTS comments_by_video (
        video_id     int,
        user_login   varchar,
        comment_dat  timestamp,
        comment      varchar,
        PRIMARY KEY (video_id, user_login)
    );
   (aucune ligne retournée / OK)

CQL> CREATE TABLE IF NOT EXISTS comments_by_user (
        user_login   varchar,
        video_i

3. Exécuter les insertions suivantes :

```sql
INSERT INTO users(login, firstname, lastname) VALUES ('cschmidt', 'chris', 'SCHMIDT');
INSERT INTO users(login, firstname, lastname) VALUES ('saraj00n', 'Sarah', 'JUNE');
INSERT INTO videos (video_id, video_name, video_uploaded, user_login) VALUES (1,'Keyboard Cat', toTimestamp(now()), 'cshmidt');
INSERT INTO videos (video_id, video_name, video_uploaded, user_login) VALUES (2,'nyan cat', toTimestamp(now()), 'saraj00n');
INSERT INTO user_videos_index (login, video_id, video_name) VALUES ('cschmidt',1, 'Keyboard Cat');
INSERT INTO user_videos_index (login, video_id, video_name) VALUES ('saraj00n',2, 'nyan cat');
INSERT INTO comments_by_user (user_login, video_id, comment_dat, comment) VALUES ('saraj00n', 1, '2017-01-15 12:00+0000', 'first');
INSERT INTO comments_by_video (video_id, user_login, comment_dat, comment) VALUES (1, 'saraj00n', '2017-01-15 12:00+0000', 'first');
INSERT INTO comments_by_user (user_login, video_id, comment_dat, comment) VALUES ('MasterTroll', 1, '2017-01-15 12:05+0000', 'thats lame');
INSERT INTO comments_by_video (video_id, user_login, comment_dat, comment) VALUES (1, 'MasterTroll', '2017-01-15 12:05+0000', 'thats lame');
```

**✅ Exécution (Python)**

⚠️ Notez déjà (on y reviendra en Q4.1 et Q7) : la vidéo 1 est insérée avec
`user_login = 'cshmidt'` (**sans le premier "c"**) alors que l'utilisateur
réel s'appelle `cschmidt`. Ce n'est pas corrigé ici : cette incohérence de
saisie fait partie de l'exercice.

In [ ]:
inserts = [
    "INSERT INTO users(login, firstname, lastname) VALUES ('cschmidt', 'chris', 'SCHMIDT');",
    "INSERT INTO users(login, firstname, lastname) VALUES ('saraj00n', 'Sarah', 'JUNE');",
    "INSERT INTO videos (video_id, video_name, video_uploaded, user_login) VALUES (1,'Keyboard Cat', toTimestamp(now()), 'cshmidt');",
    "INSERT INTO videos (video_id, video_name, video_uploaded, user_login) VALUES (2,'nyan cat', toTimestamp(now()), 'saraj00n');",
    "INSERT INTO user_videos_index (login, video_id, video_name) VALUES ('cschmidt',1, 'Keyboard Cat');",
    "INSERT INTO user_videos_index (login, video_id, video_name) VALUES ('saraj00n',2, 'nyan cat');",
    "INSERT INTO comments_by_user (user_login, video_id, comment_dat, comment) VALUES ('saraj00n', 1, '2017-01-15 12:00+0000', 'first');",
    "INSERT INTO comments_by_video (video_id, user_login, comment_dat, comment) VALUES (1, 'saraj00n', '2017-01-15 12:00+0000', 'first');",
    "INSERT INTO comments_by_user (user_login, video_id, comment_dat, comment) VALUES ('MasterTroll', 1, '2017-01-15 12:05+0000', 'thats lame');",
    "INSERT INTO comments_by_video (video_id, user_login, comment_dat, comment) VALUES (1, 'MasterTroll', '2017-01-15 12:05+0000', 'thats lame');",
]
for i in inserts:
    run_cql(i)


4. Faire une requête pour visualiser le contenu de chaque table.

In [ ]:
for t in ["users", "videos", "user_videos_index", "comments_by_video", "comments_by_user"]:
    print(f"=== {t} ===")
    run_cql(f"SELECT * FROM {t};")


4. 1. Que constatez vous ? (N'hésitez pas expliquer ou demander au prof !)

**✅ Réponse**

- Les mêmes informations apparaissent **dupliquées dans plusieurs tables**
  (ex. `video_name` dans `videos` ET dans `user_videos_index`, le login et le
  commentaire dans `comments_by_video` ET `comments_by_user`) : c'est la
  **dénormalisation** typique de Cassandra, chaque table étant construite
  pour une seule requête.
- La table `videos` contient `user_login = 'cshmidt'` pour la vidéo 1, alors
  que la table `users` ne connaît que `cschmidt`, et que
  `user_videos_index` utilise bien `cschmidt`. **Rien n'empêche cette
  incohérence** : Cassandra n'a ni clé étrangère ni contrainte d'intégrité
  référentielle, contrairement à une base relationnelle. Si `cshmidt` avait
  été une vraie faute de frappe en production, elle serait passée inaperçue
  jusqu'à ce qu'une requête ne retrouve pas les données attendues (voir
  question 7).

5. Qu’identifiez-vous au niveau des tables ``comments_by_video`` et ``comments_by_user`` par rapport à la table users ? Corriger le problème.

**✅ Réponse**

Les tables `comments_by_video` et `comments_by_user` ne stockent que
`user_login` : pour afficher un commentaire avec le prénom/nom de son auteur,
il faudrait faire une **requête supplémentaire** sur la table `users` pour
chaque commentaire (Cassandra ne sait pas faire de jointure). C'est
exactement le problème que la dénormalisation est censée éviter.

**Correction** : dupliquer (dénormaliser) `firstname`/`lastname` directement
dans les deux tables de commentaires, pour rendre chaque requête
autosuffisante :

In [ ]:
fixes = [
    "ALTER TABLE comments_by_video ADD user_firstname varchar;",
    "ALTER TABLE comments_by_video ADD user_lastname varchar;",
    "ALTER TABLE comments_by_user ADD user_firstname varchar;",
    "ALTER TABLE comments_by_user ADD user_lastname varchar;",
]
for f in fixes:
    run_cql(f)

# on met à jour les lignes déjà insérées pour qu'elles portent l'info dénormalisée
updates = [
    "UPDATE comments_by_video SET user_firstname='Sarah', user_lastname='JUNE' WHERE video_id=1 AND user_login='saraj00n';",
    "UPDATE comments_by_user  SET user_firstname='Sarah', user_lastname='JUNE' WHERE user_login='saraj00n' AND video_id=1;",
]
for u in updates:
    run_cql(u)

run_cql("SELECT * FROM comments_by_video;")
run_cql("SELECT * FROM comments_by_user;")


6. Faire une requête permettant d’afficher les attributs de la vidéo numéro 1.

In [ ]:
run_cql("SELECT * FROM videos WHERE video_id = 1;")

7. Faire une requête permettant d’afficher tous les id des vidéos insérées par la personne de login cshmidt.   

**✅ Réponse**

Une recherche par `user_login` directement sur `videos` échoue tant qu'il
n'y a pas d'index (`user_login` n'est pas dans la clé primaire) :

In [ ]:
# Sans index -> Cassandra refuse (colonne non indexée, hors clé primaire)
run_cql("SELECT video_id FROM videos WHERE user_login = 'cshmidt';")


8. Créer un index sur l’attribut user_login de la table videos et refaire la requête de la question précédente.

In [ ]:
run_cql("CREATE INDEX IF NOT EXISTS videos_user_login_idx ON videos (user_login);")

# la requête fonctionne maintenant sans ALLOW FILTERING
run_cql("SELECT video_id FROM videos WHERE user_login = 'cshmidt';")

# à titre de comparaison : avec la bonne orthographe, rien ne remonte !
# (nouvelle confirmation de l'incohérence relevée en 4.1)
run_cql("SELECT video_id FROM videos WHERE user_login = 'cschmidt';")


9. Que se passe-t-il lorsque l’on fait l’insertion suivante :  
> INSERT INTO users (login, firstname, lastname) VALUES ('saraj00n', 'Sarah', 'Junior');

In [ ]:
run_cql("INSERT INTO users (login, firstname, lastname) VALUES ('saraj00n', 'Sarah', 'Junior');")
run_cql("SELECT * FROM users WHERE login='saraj00n';")


**✅ Réponse**

Aucune erreur : en Cassandra, `INSERT` est en réalité un **upsert**. Comme
`login` est la clé primaire de `users` et que `saraj00n` existe déjà, la
ligne est simplement **écrasée** : `lastname` passe silencieusement de
`'JUNE'` à `'Junior'`. Il n'y a pas de contrainte d'unicité qui lèverait une
erreur (contrairement à une clé primaire relationnelle) — la dernière
écriture gagne (*last write wins*, basé sur un timestamp interne).

10. Modifier la table videos pour ajouter une colonne production de type ``SET<TEXT>``  (cf. [Doc](https://cassandra.apache.org/doc/latest/cql/ddl.html#alter-table) ).

Pour visualiser le schéma de la table :

> DESCRIBE TABLE videos ;

In [ ]:
run_cql("ALTER TABLE videos ADD production set<text>;")
run_cql("DESCRIBE TABLE videos;")


11. Faire une mise à jour sur une video avec l'id 1 en mettant a jour la production  _'Walt Disney Pictures'_ et  __'Roth Films'_   suivante :
 > UPDATE videos SET production = { 'Walt Disney Pictures',  'Roth Films' } WHERE video_id = 1;

In [ ]:
run_cql("UPDATE videos SET production = { 'Walt Disney Pictures', 'Roth Films' } WHERE video_id = 1;")
run_cql("SELECT video_id, video_name, production FROM videos WHERE video_id=1;")


12. Ajouter un nouveau producteur à la vidéo numéro 1.

**✅ Réponse**

On utilise l'opérateur `+` pour **ajouter** un élément à un `SET` existant
sans réécrire toute la collection :

In [ ]:
run_cql("UPDATE videos SET production = production + {'Universal Pictures'} WHERE video_id = 1;")
run_cql("SELECT video_id, production FROM videos WHERE video_id=1;")


13. Faire une requête sur la table videos pour sélectionner les vidéos dont l’attribut production contient (mot-clé ``CONTAINS``) le producteur que vous venez d’ajouter

**✅ Réponse**

Comme pour toute colonne hors clé primaire, `CONTAINS` sur une collection a
besoin soit de `ALLOW FILTERING`, soit d'un index sur la collection. On
choisit la solution propre (index) :

In [ ]:
run_cql("CREATE INDEX IF NOT EXISTS videos_production_idx ON videos (production);")
run_cql("SELECT video_id, video_name, production FROM videos WHERE production CONTAINS 'Universal Pictures';")


14. Modifier la table users pour ajouter un attribut children de type ``MAP<TEXT,INT>`` pour représenter des couples (prénom d’enfant, âge)

In [ ]:
run_cql("ALTER TABLE users ADD children map<text,int>;")
run_cql("DESCRIBE TABLE users;")


14. _Bis :_  Mettre a jour les enfants de l'utilisateur ``cschmidt`` avec cette valeur {'Tom':4,'Sofia':12}

In [ ]:
run_cql("UPDATE users SET children = {'Tom':4,'Sofia':12} WHERE login='cschmidt';")
run_cql("SELECT login, children FROM users WHERE login='cschmidt';")


15. Faire une requête permettant de récupérer l’utilisateur ayant comme enfant ‘Tom’

**✅ Réponse**

Pour filtrer sur une **clé** de `MAP`, on indexe les clés avec `KEYS(...)` :

In [ ]:
run_cql("CREATE INDEX IF NOT EXISTS users_children_keys_idx ON users (KEYS(children));")
run_cql("SELECT login, firstname, lastname, children FROM users WHERE children CONTAINS KEY 'Tom';")


16. Faire une requête permettant de récupérer l’utilisateur ayant un enfant de 12 ans

**✅ Réponse**

Pour filtrer sur une **valeur** de `MAP`, on indexe les valeurs avec
`VALUES(...)` (c'est d'ailleurs le comportement par défaut d'un index sur une
map) :

In [ ]:
run_cql("CREATE INDEX IF NOT EXISTS users_children_values_idx ON users (VALUES(children));")
run_cql("SELECT login, firstname, lastname, children FROM users WHERE children CONTAINS 12;")


17. Créer le type suivant et ajouter un attribut ``address`` de ce type dans la table ``users`` :

```sql
CREATE TYPE ADDRESS (
      street TEXT,
      city TEXT,
      state TEXT,
      postal_code TEXT
      );

# votre réponse pour l'ajout
```

In [ ]:
run_cql("""CREATE TYPE IF NOT EXISTS address (
    street TEXT,
    city TEXT,
    state TEXT,
    postal_code TEXT
);""")

# réponse pour l'ajout du champ à la table users :
run_cql("ALTER TABLE users ADD address frozen<address>;")


18. Ajouter l'adresse suivant a l'utilisateur ``cschmidt`` :
 > { street: '1100 Congress Ave', city: 'Austin', state: 'Texas', postal_code: '78701' }

In [ ]:
run_cql("""UPDATE users
SET address = {street: '1100 Congress Ave', city: 'Austin', state: 'Texas', postal_code: '78701'}
WHERE login = 'cschmidt';""")


19. Vérifier la modification :

In [ ]:
run_cql("SELECT login, firstname, lastname, address FROM users WHERE login='cschmidt';")

19. bis: récupérer le lastname de l'utilisateur avec l'adresse  
>  { street: '1100 Congress Ave', city: 'Austin', state: 'Texas', postal_code: '78701' }

**✅ Réponse**

`address` étant un UDT `frozen` non indexé, on ne peut filtrer que sur sa
**valeur complète**, et cela nécessite `ALLOW FILTERING` (parcours de la
table car il n'y a pas d'index) :

In [ ]:
run_cql("""SELECT lastname FROM users
WHERE address = {street: '1100 Congress Ave', city: 'Austin', state: 'Texas', postal_code: '78701'}
ALLOW FILTERING;""")


20. Exécuter les instructions suivantes puis exécuter à nouveau les requêtes de la question précédentes (en remplaçant address par address2).

```sql
ALTER TABLE users ADD address2 frozen<ADDRESS>;
UPDATE users  SET address2 = { street: '1100 Congress Ave', city: 'Austin', state: 'Texas', postal_code: '78701' } WHERE login='cschmidt';
```

In [ ]:
run_cql("ALTER TABLE users ADD address2 frozen<address>;")
run_cql("""UPDATE users SET address2 = { street: '1100 Congress Ave', city: 'Austin', state: 'Texas', postal_code: '78701' }
WHERE login='cschmidt';""")

# requête de la question 19bis, avec address2 à la place de address :
run_cql("""SELECT lastname FROM users
WHERE address2 = {street: '1100 Congress Ave', city: 'Austin', state: 'Texas', postal_code: '78701'}
ALLOW FILTERING;""")


21. Que pouvez vous dire?

**✅ Réponse**

- Les deux requêtes renvoient exactement le même résultat (`SCHMIDT`) :
  `address` et `address2` sont deux colonnes **indépendantes** de même type
  `frozen<address>`, et ce type est comparé **par valeur complète** — il
  n'est pas possible de filtrer sur un seul sous-champ (`city`, `state`...)
  d'un UDT `frozen` sans le ré-écrire en entier.
- Ajouter `address2` n'a eu **aucun impact** sur `address` : plusieurs
  colonnes peuvent partager le même type utilisateur (`ADDRESS`) sans
  interférer entre elles.
- Dans les deux cas, la requête a nécessité `ALLOW FILTERING` car ni
  `address` ni `address2` ne sont indexées : cela déclenche un balayage de
  table, ce qui est acceptable ici (jeu de données minuscule) mais
  **désastreux en production**. C'est un nouveau rappel du principe
  « *query-first design* » vu en partie 1 : si l'application a réellement
  besoin de retrouver un utilisateur par adresse, il faudrait créer une
  table dédiée `users_by_address` plutôt que de filtrer.

---
## Récapitulatif

| Fichier | Rôle |
|---|---|
| `docker-compose.yml` | Démarre Cassandra + un Jupyter pré-connecté (`cassandra-driver` installé) |
| `Dockerfile.jupyter` | Image Jupyter utilisée par le service `jupyter` du compose |
| `cassandra_TP01-solution.ipynb` (ce fichier) | Solution complète du TP, à placer dans `./notebooks/` |

Pour tout relancer proprement :
```bash
docker compose down -v   # supprime aussi les données persistées
docker compose up -d --build
```
